# PS S6E4 - GBDT TE top4 minimum

目的:
- high-IV OVR scan の top4 categorical columns を起点に、
  leak-free な最小 TE を追加して GBDT で効くかを確認する
- まずは単列 TE のみ
- pair TE / CATNUM pair / Optuna はまだやらない

対象カテゴリ:
- Crop_Growth_Stage
- Water_Source
- Crop_Type
- Irrigation_Type

## 1. Imports / config

In [1]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")


class CFG:
    COMP_NAME = "playground-series-s6e4"
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_SPLITS = 5
    INNER_SPLITS = 5

    EXP_NAME = "catboost_te_top4_min"
    SAVE_DIR = Path("/kaggle/working/exp_20260406_014_gbdt_te_top4_min")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    HIGH_IV_TOP4 = [
        "Crop_Growth_Stage",
        "Water_Source",
        "Crop_Type",
        "Irrigation_Type",
    ]

    BASE_CAT_COLS = [
        "Soil_Type",
        "Crop_Type",
        "Crop_Growth_Stage",
        "Season",
        "Irrigation_Type",
        "Water_Source",
        "Region",
        "Mulching_Used",
    ]

    TARGET_CLASSES = ["Low", "Medium", "High"]

    TE_FILLNA = 0.0
    TE_SUFFIX = "__te"

    CATBOOST_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "iterations": 3000,
        "learning_rate": 0.03,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "subsample": 0.9,
        "bootstrap_type": "Bernoulli",
        "random_seed": SEED,
        "verbose": 200,
        "allow_writing_files": False,
        "task_type": "CPU",
    }


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(CFG.SEED)
print(CFG.SAVE_DIR)

/kaggle/working/exp_20260406_014_gbdt_te_top4_min


## 2. Load data

In [2]:
INPUT_DIR = Path(f"/kaggle/input/competitions/{CFG.COMP_NAME}")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")

print(train.shape, test.shape)
display(train.head())
print(train[CFG.TARGET_COL].value_counts(dropna=False))

(630000, 21) (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


## 3. TE utility

In [3]:
def make_binary_target(y: pd.Series, positive_class: str) -> pd.Series:
    return (y == positive_class).astype(float)


def fit_te_mapping(
    x_cat: pd.Series,
    y_bin: pd.Series,
):
    tmp = pd.DataFrame({"cat": x_cat.astype("object"), "y": y_bin.values})
    global_mean = float(tmp["y"].mean())
    mapping = tmp.groupby("cat")["y"].mean().to_dict()
    return mapping, global_mean


def apply_te_mapping(
    x_cat: pd.Series,
    mapping: dict,
    global_mean: float,
    fillna_value: float = 0.0,
):
    te = x_cat.astype("object").map(mapping).fillna(global_mean)
    te = te.fillna(fillna_value)
    return te.astype(np.float32)


def build_multiclass_te_features_leakfree(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    X_test: pd.DataFrame,
    te_cols: list[str],
    target_classes: list[str],
    inner_splits: int = 5,
    seed: int = 42,
    fillna_value: float = 0.0,
):
    """
    outer train に対して inner CV で OOF TE を作る。
    valid / test は outer train 全体 fit の map を使う。
    """

    X_train_te = pd.DataFrame(index=X_train.index)
    X_valid_te = pd.DataFrame(index=X_valid.index)
    X_test_te = pd.DataFrame(index=X_test.index)

    inner_skf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    for col in te_cols:
        for cls in target_classes:
            feat_name = f"{col}{CFG.TE_SUFFIX}_{cls}"

            y_bin_full = make_binary_target(y_train, cls)

            # OOF for outer train
            oof_te = pd.Series(index=X_train.index, dtype=np.float32)

            for inner_tr_idx, inner_va_idx in inner_skf.split(X_train, y_train):
                x_tr_cat = X_train.iloc[inner_tr_idx][col]
                y_tr_bin = y_bin_full.iloc[inner_tr_idx]

                x_va_cat = X_train.iloc[inner_va_idx][col]

                mapping, global_mean = fit_te_mapping(x_tr_cat, y_tr_bin)
                oof_te.iloc[inner_va_idx] = apply_te_mapping(
                    x_va_cat, mapping, global_mean, fillna_value=fillna_value
                ).values

            # full fit for valid / test
            full_mapping, full_global_mean = fit_te_mapping(X_train[col], y_bin_full)

            valid_te = apply_te_mapping(
                X_valid[col], full_mapping, full_global_mean, fillna_value=fillna_value
            )
            test_te = apply_te_mapping(
                X_test[col], full_mapping, full_global_mean, fillna_value=fillna_value
            )

            X_train_te[feat_name] = oof_te.values
            X_valid_te[feat_name] = valid_te.values
            X_test_te[feat_name] = test_te.values

    return X_train_te, X_valid_te, X_test_te

## 4. CV function

In [4]:
def fit_catboost_cv_with_te(train_df, test_df, feature_cols_base, cat_features_base):
    y = train_df[CFG.TARGET_COL].copy()

    n_classes = y.nunique()
    oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
    test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)

    fold_scores = []
    class_names = None
    te_feature_names_final = None

    skf = StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.SEED)

    X_all = train_df[feature_cols_base].copy()
    X_test_base = test_df[feature_cols_base].copy()

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y), 1):
        X_tr_base = X_all.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()

        X_va_base = X_all.iloc[va_idx].copy()
        y_va = y.iloc[va_idx].copy()

        X_te_base = X_test_base.copy()

        X_tr_te, X_va_te, X_te_te = build_multiclass_te_features_leakfree(
            X_train=X_tr_base,
            y_train=y_tr,
            X_valid=X_va_base,
            X_test=X_te_base,
            te_cols=CFG.HIGH_IV_TOP4,
            target_classes=CFG.TARGET_CLASSES,
            inner_splits=CFG.INNER_SPLITS,
            seed=CFG.SEED,
            fillna_value=CFG.TE_FILLNA,
        )

        te_feature_names = X_tr_te.columns.tolist()
        te_feature_names_final = te_feature_names

        X_tr = pd.concat([X_tr_base.reset_index(drop=True), X_tr_te.reset_index(drop=True)], axis=1)
        X_va = pd.concat([X_va_base.reset_index(drop=True), X_va_te.reset_index(drop=True)], axis=1)
        X_te = pd.concat([X_te_base.reset_index(drop=True), X_te_te.reset_index(drop=True)], axis=1)

        final_feature_cols = X_tr.columns.tolist()
        final_cat_features = [c for c in cat_features_base if c in final_feature_cols]

        model = CatBoostClassifier(**CFG.CATBOOST_PARAMS)

        model.fit(
            X_tr,
            y_tr.reset_index(drop=True),
            cat_features=final_cat_features,
            eval_set=(X_va, y_va.reset_index(drop=True)),
            use_best_model=True,
        )

        if class_names is None:
            class_names = list(model.classes_)

        va_pred = model.predict(X_va).reshape(-1)
        va_proba = model.predict_proba(X_va)
        te_proba = model.predict_proba(X_te)

        score = balanced_accuracy_score(y_va, va_pred)
        fold_scores.append(score)

        oof_proba[va_idx] = va_proba
        test_proba += te_proba / CFG.N_SPLITS

        print(
            f"fold {fold} balanced_accuracy = {score:.6f} | "
            f"n_base={len(feature_cols_base)} n_te={len(te_feature_names)} n_total={len(final_feature_cols)}"
        )

        del model, X_tr, X_va, X_te, X_tr_te, X_va_te, X_te_te
        gc.collect()

    cv_score = float(np.mean(fold_scores))
    return oof_proba, test_proba, fold_scores, cv_score, class_names, te_feature_names_final

## 5. Base feature list

In [5]:
feature_cols_base = [c for c in train.columns if c not in [CFG.ID_COL, CFG.TARGET_COL]]
cat_features_base = [c for c in CFG.BASE_CAT_COLS if c in feature_cols_base]

print("n_base_features =", len(feature_cols_base))
print("cat_features_base =", cat_features_base)

n_base_features = 19
cat_features_base = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Region', 'Mulching_Used']


## 6. Run CV

In [6]:
oof_proba, test_proba, fold_scores, cv_score, class_names, te_feature_names = fit_catboost_cv_with_te(
    train_df=train,
    test_df=test,
    feature_cols_base=feature_cols_base,
    cat_features_base=cat_features_base,
)

print("class_names =", class_names)
print("fold_scores =", fold_scores)
print("cv_score =", cv_score)
print("n_te_features =", len(te_feature_names))
print("te_feature_names[:12] =", te_feature_names[:12])

0:	learn: 0.9801896	test: 0.9804778	best: 0.9804778 (0)	total: 1.16s	remaining: 58m 7s
200:	learn: 0.9839069	test: 0.9841048	best: 0.9841379 (198)	total: 3m 22s	remaining: 47m 4s
400:	learn: 0.9841048	test: 0.9842599	best: 0.9842676 (373)	total: 6m 24s	remaining: 41m 34s
600:	learn: 0.9842054	test: 0.9843002	best: 0.9843160 (527)	total: 9m 21s	remaining: 37m 23s
800:	learn: 0.9843053	test: 0.9843314	best: 0.9843397 (782)	total: 12m 32s	remaining: 34m 24s
1000:	learn: 0.9843795	test: 0.9843793	best: 0.9843793 (961)	total: 15m 46s	remaining: 31m 29s
1200:	learn: 0.9844743	test: 0.9843716	best: 0.9843954 (1063)	total: 19m	remaining: 28m 28s
1400:	learn: 0.9845640	test: 0.9843633	best: 0.9843954 (1063)	total: 22m 12s	remaining: 25m 20s
1600:	learn: 0.9846348	test: 0.9843643	best: 0.9843954 (1063)	total: 25m 36s	remaining: 22m 22s
1800:	learn: 0.9847054	test: 0.9844362	best: 0.9844362 (1797)	total: 28m 59s	remaining: 19m 17s
2000:	learn: 0.9847874	test: 0.9844525	best: 0.9844525 (2000)	tota

## 7. Save artifacts

In [7]:
np.save(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_proba.npy", oof_proba)
np.save(CFG.SAVE_DIR / f"pred_{CFG.EXP_NAME}_proba.npy", test_proba)

oof_pred_idx = oof_proba.argmax(axis=1)
oof_pred_label = pd.Series([class_names[i] for i in oof_pred_idx])

oof_df = pd.DataFrame({
    CFG.ID_COL: train[CFG.ID_COL],
    "y_true": train[CFG.TARGET_COL],
    "y_pred": oof_pred_label,
})
oof_df.to_csv(CFG.SAVE_DIR / f"oof_{CFG.EXP_NAME}_labels.csv", index=False)

sub = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET_COL: [class_names[i] for i in test_proba.argmax(axis=1)]
})
sub.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

feature_columns_df = pd.DataFrame({
    "feature": feature_cols_base + te_feature_names,
    "is_base_cat": [c in cat_features_base for c in (feature_cols_base + te_feature_names)],
    "is_te": [c in te_feature_names for c in (feature_cols_base + te_feature_names)],
})
feature_columns_df.to_csv(CFG.SAVE_DIR / "feature_columns.csv", index=False)

with open(CFG.SAVE_DIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "cv_score": cv_score,
            "fold_scores": fold_scores,
            "class_names": class_names,
            "n_base_features": len(feature_cols_base),
            "n_cat_features_base": len(cat_features_base),
            "n_te_features": len(te_feature_names),
            "n_total_features": len(feature_cols_base) + len(te_feature_names),
            "high_iv_top4": CFG.HIGH_IV_TOP4,
            "target_classes": CFG.TARGET_CLASSES,
            "inner_splits": CFG.INNER_SPLITS,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260406_014_gbdt_te_top4_min


## 8. Quick summary

In [8]:
summary_df = pd.DataFrame({
    "item": [
        "cv_score",
        "n_base_features",
        "n_cat_features_base",
        "n_te_features",
        "n_total_features",
    ],
    "value": [
        cv_score,
        len(feature_cols_base),
        len(cat_features_base),
        len(te_feature_names),
        len(feature_cols_base) + len(te_feature_names),
    ]
})
display(summary_df)

,item,value
0,cv_score,0.961534
1,n_base_features,19.000000
2,n_cat_features_base,8.000000
3,n_te_features,12.000000
4,n_total_features,31.000000
